In [1]:
!pwd

/home/ylnner/Documents/KD_gradient_boosting/scripts


In [15]:
STUDENT_HIDDEN  = (64, 32)
STUDENT_DROPOUT = 0.1

In [ ]:
import os
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

class StudentModel(nn.Module):
    def __init__(self, n_features = 8, hidden_dims=STUDENT_HIDDEN, dropout=STUDENT_DROPOUT):
        super().__init__()

        # Capas ocultas: Linear -> ReLU -> Dropout
        layers = []
        in_dim = n_features
        for h in hidden_dims:
            layers += [nn.Linear(in_dim, h), nn.ReLU(), nn.Dropout(dropout)]
            in_dim = h
        
        # Se agrega una capa de salida. 
        layers.append(nn.Linear(in_dim, 1))

        self.mlp = nn.Sequential(*layers)


    def forward(self, x):
        return self.mlp(x).squeeze(-1)

In [ ]:
import re
main_path = "/home/ylnner/Documents/KD_gradient_boosting/my_final_version/results"
N_list = [60, 100, 200, 300]
f_values = []

for N in N_list:
    str_N = "N_" + str(N)
    best_models_path = os.path.join(main_path, str_N, "student/best_models")
    
    if not os.path.isdir(best_models_path):
        print(f"Warning: Directory not found: {best_models_path}")
    else:
        print("best_models_path: ", best_models_path)
        for file in os.listdir(best_models_path):
            if file.endswith(".pth"):
                match = re.match(r"^f(\d+\.\d+)_", file)

            if match:
                f_value = str(match.group(1))
                #f_values.append(f_value)
                #print(f"Found f value: {f_value} in file: {file}")

                # mlp_kd_005_scripted
                final_name = "mlp_kd_"
                if f_value == "0.05":
                    final_name += "005_scripted"
                elif f_value == "0.10":
                    final_name += "010_scripted"
                elif f_value == "0.15":
                    final_name += "015_scripted"
                elif f_value == "1.00":
                    final_name += "100_scripted"

                print(f"Final name for f={f_value}: {final_name}")
                
                model = StudentModel()
                model_path = os.path.join(best_models_path, file)
                print(f"Loading model from: {model_path}")

                weights = torch.load(model_path, map_location=torch.device('cpu'))
                model.load_state_dict(weights)
                model.eval()

                total_params = sum(p.numel() for p in model.parameters())
                print(f"Total parameters: {total_params}")

                scripted_model = torch.jit.script(model)
                scripted_model.save(f"{final_name}.pt")
                print("Conversion complete!")

best_models_path:  /home/ylnner/Documents/KD_gradient_boosting/my_final_version/results/N_60/student/best_models
Final name for f=0.10: mlp_kd_010_scripted
Loading model from: /home/ylnner/Documents/KD_gradient_boosting/my_final_version/results/N_60/student/best_models/f0.10_student_a0.1_T2_s0.pth
Total parameters: 2689
Conversion complete!
Final name for f=0.15: mlp_kd_015_scripted
Loading model from: /home/ylnner/Documents/KD_gradient_boosting/my_final_version/results/N_60/student/best_models/f0.15_student_a0.2_T8_s5.pth
Total parameters: 2689
Conversion complete!
Final name for f=1.00: mlp_kd_100_scripted
Loading model from: /home/ylnner/Documents/KD_gradient_boosting/my_final_version/results/N_60/student/best_models/f1.00_student_a0.3_T4_s0.pth
Total parameters: 2689
Conversion complete!
Final name for f=0.05: mlp_kd_005_scripted
Loading model from: /home/ylnner/Documents/KD_gradient_boosting/my_final_version/results/N_60/student/best_models/f0.05_student_a0_T1_s7.pth
Total paramet

In [11]:

model_path = 'best_models/bilstm_best_seed8.pth'
print("*"*50)
print("BILSTM MODEL")
print("*"*50)
# BiLSTM best config
best_config = {'seq_length': 8, 'hidden_size': 32, 'num_layers': 3, 'dropout': 0.2, 'lr': 0.0005, 'batch_size': 32}
model = LoraCollisionLSTM(hidden_size=best_config['hidden_size'], num_layers=best_config['num_layers'], dropout=best_config['dropout'], add_sigmoidal=False)
weights = torch.load(model_path, map_location=torch.device('cpu'))
model.load_state_dict(weights)
model.eval()

total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params}")

scripted_model = torch.jit.script(model)
scripted_model.save("best_models/bilstm_scripted.pt")
print("Conversion complete!")


**************************************************
BILSTM MODEL
**************************************************
Total parameters: 63297
Conversion complete!


In [15]:
print("*"*50)
print("TRANSFORMER MODEL")
print("*"*50)
# Transfromer best config
#model_path = 'best_models/transformer_best_seed0.pth'
model_path = 'best_models/transformer_best_seed2.pth'

best_config = {'seq_length': 8, 'd_model': 32, 'n_heads': 4, 'n_layers': 1, 'dropout': 0.2, 'lr': 0.0005, 'batch_size': 32}
model_transformer = LoraCollisionTransformer(num_numerical_features=9, d_model=best_config['d_model'], n_heads=best_config['n_heads'], n_layers=best_config['n_layers'], dropout=best_config['dropout'], add_sigmoid=False)

weights = torch.load(model_path, map_location=torch.device('cpu'))
model_transformer.load_state_dict(weights)

model_transformer.eval()
total_params = sum(p.numel() for p in model_transformer.parameters())
print(f"Total parameters: {total_params}")
scripted_model = torch.jit.script(model_transformer)
scripted_model.save("best_models/transformer_scripted.pt")
print("Conversion complete!")

**************************************************
TRANSFORMER MODEL
**************************************************
Total parameters: 14113
Conversion complete!


In [ ]:
!ls

In [ ]:
import torch
from TwoBranchLoraCollisionTransformer import TwoBranchLoraCollisionTransformer
print("*"*50)
print("TWO BRANCHES MODEL")
print("*"*50)
# Transfromer best config
#model_path = 'best_models/transformer_best_seed0.pth'
model_path = 'best_models/two_branch_transformer_best_seed9.pth'

best_config = {'seq_length': 8, 'd_model': 32, 'n_heads': 2, 'n_layers': 1, 'dropout': 0.1, 'lr': 0.0005, 'batch_size': 32, 'fusion_hidden': 32}
model_transformer = TwoBranchLoraCollisionTransformer(d_model=best_config['d_model'], n_heads=best_config['n_heads'], n_layers=best_config['n_layers'], dropout=best_config['dropout'], static_hidden_dims=(64, 32), fusion_hidden=best_config['fusion_hidden'], add_sigmoid=False)

weights = torch.load(model_path, map_location=torch.device('cpu'))
model_transformer.load_state_dict(weights)

model_transformer.eval()
total_params = sum(p.numel() for p in model_transformer.parameters())
print(f"Total parameters: {total_params}")
scripted_model = torch.jit.script(model_transformer)
scripted_model.save("best_models/two_branch_transformer_scripted.pt")
print("Conversion complete!")

# Scaler Bilstm

In [12]:
import json
import joblib  # Using joblib to match the actual file formatting

# Load your scaler using joblib instead of pickle
# Pass the filename directly to joblib.load
model_path = 'best_models/bilstm_scaler_seed8.pkl'
scaler = joblib.load(model_path)

# Extract the mathematical attributes
# Note: If this is a MinMaxScaler instead of StandardScaler, 
# you'll want to use: "min": scaler.data_min_.tolist(), "max": scaler.data_max_.tolist()
scaler_data = {
    "mean": scaler.mean_.tolist(),
    "scale": scaler.scale_.tolist()
}

# Save it to a clean JSON file for OMNeT++
with open("scaler_params.json", "w") as f:
    json.dump(scaler_data, f)

print("Extraction successful!")
scaler_data

Extraction successful!


/home/ylnner/miniconda3/envs/p_new/lib/python3.14/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.7.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


{'mean': [48.65901333333334,
  12.03978895,
  9.818097488670633,
  0.020066399999999998,
  10.933333333333334,
  -447.96115595273824,
  715.0,
  214.82142857142858],
 'scale': [8.67688779738962,
  13.329355167546092,
  17.71640489314641,
  0.0037267134099632617,
  0.813770374382247,
  14642.787636931365,
  88.94219631712659,
  75.23750659052429]}

In [13]:
import json
import joblib  # Using joblib to match the actual file formatting

# Load your scaler using joblib instead of pickle
# Pass the filename directly to joblib.load
model_path = 'best_models/bilstm_scaler_delta_t_seed8.pkl'
scaler = joblib.load(model_path) 


# Extract the mathematical attributes
# Note: If this is a MinMaxScaler instead of StandardScaler, 
# you'll want to use: "min": scaler.data_min_.tolist(), "max": scaler.data_max_.tolist()
scaler_data = {
    "mean": scaler.mean_.tolist(),
    "scale": scaler.scale_.tolist()
}

# Save it to a clean JSON file for OMNeT++
with open("scaler_params.json", "w") as f:
    json.dump(scaler_data, f)

print("Extraction successful!")
scaler_data

Extraction successful!


{'mean': [61.01802962820513], 'scale': [40.76300672650946]}

# Scaler Transformer

In [9]:
import json
import joblib  # Using joblib to match the actual file formatting

# Load your scaler using joblib instead of pickle
# Pass the filename directly to joblib.load
model_path = 'best_models/transformer_scaler_seed2.pkl'


scaler = joblib.load(model_path) 


# Extract the mathematical attributes
# Note: If this is a MinMaxScaler instead of StandardScaler, 
# you'll want to use: "min": scaler.data_min_.tolist(), "max": scaler.data_max_.tolist()
scaler_data = {
    "mean": scaler.mean_.tolist(),
    "scale": scaler.scale_.tolist()
}

# Save it to a clean JSON file for OMNeT++
with open("scaler_params.json", "w") as f:
    json.dump(scaler_data, f)

print("Extraction successfulll!")
scaler_data

Extraction successfulll!


{'mean': [48.55624,
  11.995906750000001,
  9.738305531309523,
  0.0206337,
  11.033333333333333,
  -637.2126803374404,
  715.0,
  214.82142857142858],
 'scale': [8.63089616932873,
  13.408186415520877,
  17.75413475236267,
  0.003885927908492385,
  0.8359957469322968,
  14639.724221620067,
  88.94219631712659,
  75.23750659052429]}

In [10]:
import json
import joblib  # Using joblib to match the actual file formatting

# Load your scaler using joblib instead of pickle
# Pass the filename directly to joblib.load
model_path = 'best_models/transformer_scaler_delta_t_seed2.pkl'
scaler = joblib.load(model_path) 


# Extract the mathematical attributes
# Note: If this is a MinMaxScaler instead of StandardScaler, 
# you'll want to use: "min": scaler.data_min_.tolist(), "max": scaler.data_max_.tolist()
scaler_data = {
    "mean": scaler.mean_.tolist(),
    "scale": scaler.scale_.tolist()
}

# Save it to a clean JSON file for OMNeT++
with open("scaler_params.json", "w") as f:
    json.dump(scaler_data, f)

print("Extraction successful!")
scaler_data

Extraction successful!


{'mean': [61.31933958104395], 'scale': [40.97636157062998]}

# Scaler TwoBranches

In [ ]:
import json
import joblib  # Using joblib to match the actual file formatting

# Load your scaler using joblib instead of pickle
# Pass the filename directly to joblib.load
model_path = 'best_models/two_branch_transformer_scaler_seed9.pkl'


scaler = joblib.load(model_path) 


# Extract the mathematical attributes
# Note: If this is a MinMaxScaler instead of StandardScaler, 
# you'll want to use: "min": scaler.data_min_.tolist(), "max": scaler.data_max_.tolist()
scaler_data = {
    "mean": scaler.mean_.tolist(),
    "scale": scaler.scale_.tolist()
}

# Save it to a clean JSON file for OMNeT++
with open("scaler_params.json", "w") as f:
    json.dump(scaler_data, f)

print("Extraction successfulll!")
scaler_data

In [ ]:
import json
import joblib  # Using joblib to match the actual file formatting

# Load your scaler using joblib instead of pickle
# Pass the filename directly to joblib.load
model_path = 'best_models/two_branch_transformer_scaler_delta_t_seed9.pkl'
scaler = joblib.load(model_path) 


# Extract the mathematical attributes
# Note: If this is a MinMaxScaler instead of StandardScaler, 
# you'll want to use: "min": scaler.data_min_.tolist(), "max": scaler.data_max_.tolist()
scaler_data = {
    "mean": scaler.mean_.tolist(),
    "scale": scaler.scale_.tolist()
}

# Save it to a clean JSON file for OMNeT++
with open("scaler_params.json", "w") as f:
    json.dump(scaler_data, f)

print("Extraction successful!")
scaler_data